# Next step: step 2, names for the 22 parameters
**What it adds:** names for the 22 SPAM values, and a parser for model strings. Models stay plain arrays, so
one model is `(22,)` and a whole optimizer population is `(n_models, 22)`, with no conversions.

**How it connects:**
- **Below:** nothing changes for SPAM. `simulate` passes the array straight to `jspam.run_collision`, which
  checks there are 22 values.
- **Above:** later steps use `PARAM_NAMES` to label saved settings, and `parse_model_string` to read targets'
  model strings. A model class that bundles fixed encounter parameters with their simulate / image / score
  settings may come later, at the `simr` level.

The full, current demo is `simulator_tour.ipynb`.

**To run it** (from this repo's root, in bash or PowerShell), then open the `127.0.0.1` link it prints:
```bash
docker run --rm -it -p 8888:8888 -v "${PWD}:/gtc" -w /gtc -e UV_PROJECT_ENVIRONMENT=/tmp/venv \
    ogdenm12/gtc-base-image:v2 uv run --group notebook jupyter lab --ip=0.0.0.0 --allow-root --no-browser
```
- `-v "${PWD}:/gtc"` mounts this repo into the container, so edits on Windows show up immediately.
- `UV_PROJECT_ENVIRONMENT=/tmp/venv` keeps the Linux virtual environment inside the container, out of the repo.
  It's rebuilt each time the container starts (a minute or two, mostly compiling SPAM).
- Our package is installed in editable mode. After editing a `.py` file, restart the kernel to pick it up.

## What changed in `sim.py`
**Added:** the names, a name → position lookup, and the parser.
```python
PARAM_NAMES = (
    "x", "y", "z", "vx", "vy", "vz",
    "mass1", "mass2", "rout1", "rout2",
    "phi1", "phi2", "theta1", "theta2",
    "epsilon1", "epsilon2",
    "rscale1_1", "rscale1_2", "rscale1_3", "rscale2_1", "rscale2_2", "rscale2_3",
)
PARAM_INDEX = {name: i for i, name in enumerate(PARAM_NAMES)}


def parse_model_string(model: str) -> np.ndarray:
    values = model.strip().split(",")
    if len(values) < len(PARAM_NAMES):
        raise ValueError(f"expected at least {len(PARAM_NAMES)} values, got {len(values)}")
    return np.array(values[: len(PARAM_NAMES)], dtype=float)
```
**Unchanged:** `simulate(params, n1, n2)` still passes the array straight to SPAM. Only its docstring now
points at `PARAM_NAMES` for the order.

In [ ]:
import numpy as np

from galactic_tidal_constraints.simr import sim

## 1. The names
Position `i` in a model array is `PARAM_NAMES[i]`. `PARAM_INDEX` goes the other way.

In [ ]:
for i, name in enumerate(sim.PARAM_NAMES):
    print(f"{i:2d}  {name}")
print("mass1 is at position", sim.PARAM_INDEX["mass1"])

## 2. From a model string
A line from JSPAM's `archive/data.txt`. It has 34 values; SPAM's own parser reads the first 22 and ignores the
rest, and so does `parse_model_string`. Zipping with the names gives a readable record.

In [ ]:
MODEL_STRING = "-9.93853,-4.5805,15.43348,-1.54025,-2.97899,4.69234,34.41671,29.49607,2.97913,4.2501,301.0,35.5,310.80745,321.98757,0.3,0.3,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0,0,0.24008,-2.41999,5.62715,0.0,1.0,0.0,1.0,0.0,0.0"
print("values in the string:", len(MODEL_STRING.split(",")))

params = sim.parse_model_string(MODEL_STRING)
print("shape:", params.shape)
dict(zip(sim.PARAM_NAMES, params.tolist()))

## 3. Reading and changing values by name
Arrays can be changed in place, so copy first when the original should stay as it was.

In [ ]:
lighter = params.copy()
lighter[sim.PARAM_INDEX["mass2"]] /= 2
print("original mass2:", params[sim.PARAM_INDEX["mass2"]], " copy's mass2:", lighter[sim.PARAM_INDEX["mass2"]])

## 4. A population is just a bigger array
Many models stack into `(n_models, 22)`, and the same lookup reads one parameter across all of them. This is
the form an optimizer will work in.

In [ ]:
population = np.tile(params, (5, 1))
population[:, sim.PARAM_INDEX["mass2"]] *= np.linspace(0.5, 1.5, 5)
print("population shape:", population.shape)
print("mass2 of each model:", population[:, sim.PARAM_INDEX["mass2"]])

## 5. Wrong input is caught
Too few values in a string, or the wrong number in an array, raise an error before SPAM runs.

In [ ]:
try:
    sim.parse_model_string("1,2,3")
except ValueError as error:
    print("string ->", error)

try:
    sim.simulate([0.0] * 21)
except ValueError as error:
    print("array  ->", error)

## 6. Simulate each model in the population
`simulate` takes one row at a time. Running a whole population in parallel comes later, in `simr`.

The secondary ends near its given position in every model, because every run stops at "now". Changing
`mass2` changes the orbit *before* now, and so how strongly the primary's disk is pulled apart. The spread
of the primary's disk particles shows that (it also varies a little from run to run, since the particles
aren't seeded yet).

In [ ]:
N1 = 1000
for row in population:
    particles = sim.simulate(row, n1=N1, n2=N1)
    spread = particles[:N1, :2].std()
    print(f"mass2 = {row[sim.PARAM_INDEX['mass2']]:6.2f}   secondary ends at x, y = {particles[-1, :2].round(3)}"
          f"   primary disk spread = {spread:.2f}")